# Parte 2

O objetivo desta parte do trabalho é analisar o **comportamento dos índices** das tabelas do SGBD através do exame e análise das tabelas de estatísticas para consultas SQL sobre uma tabela criada com dados aleatórios.

Configuração inicial:

In [29]:
# Conectar ao banco de dados PostgreSQL
import psycopg2

config = {
    'dbname': 'icomp',
    'user': 'icomp',
    'password': 'icomp123',
}

conn = psycopg2.connect(**config)
cur = conn.cursor()

# Configurar rich
from rich.table import Table
from rich.console import Console

console = Console()

---
## Tarefa 5
**Preparação da Tabela Aleatória**

Criar uma tabela com uma chave simples e alguns dados de exemplo. Cada valor de chave é um número incremental e está associado a com valores que variam de 0 até 10:

In [16]:
cur.execute("""
DROP TABLE IF EXISTS t;
CREATE TABLE t (
    k SERIAL PRIMARY KEY,
    v INTEGER
);  
""")

cur.execute("""
INSERT INTO t(v)
SELECT trunc(random() * 11)      -- valores entre 0 e 10
FROM generate_series(1, 100000); -- 100 mil
""")

### O que entregar

Imprimir os valores das 10 primeiras tuplas da tabela, ordenando por k:

In [17]:
cur.execute("""
SELECT * FROM t ORDER BY k LIMIT 10;
""")
rows = cur.fetchall()

print('\n'.join(f'Tupla(k: {row[0]:>2}, v: {row[1]})' for row in rows))

Tupla(k:  1, v: 4)
Tupla(k:  2, v: 7)
Tupla(k:  3, v: 8)
Tupla(k:  4, v: 9)
Tupla(k:  5, v: 1)
Tupla(k:  6, v: 7)
Tupla(k:  7, v: 9)
Tupla(k:  8, v: 8)
Tupla(k:  9, v: 2)
Tupla(k: 10, v: 2)


---
## Tarefa 6
**Páginas criadas**

Verifique quantas páginas com blocos foram criadas para a tabela da Tarefa 5.

Comando: `SELECT relname, relpages, reltuples FROM pg_class WHERE relname='t';`

### O que entregar

Imprimir o resutlado do comando SQL

Antes de executar a consulta, é necessário atualizar as estatísticas da tabela com o comando `ANALYZE t;` para garantir que os valores retornados estejam corretos

In [ ]:
cur.execute("ANALYZE t;")
conn.commit()

In [30]:
# Executa a consulta
cur.execute("""
    SELECT relname, relpages, reltuples
    FROM pg_class
    WHERE relname = 't';
""")

name, npages, ntuples = cur.fetchall()[0]

table = Table(title="Estatísticas da Tabela t")
table.add_column("Nome", justify="left")
table.add_column("Páginas", justify="right")
table.add_column("Tuplas", justify="right")

table.add_row(str(name), str(npages), str(ntuples))

# Imprime
console.print(table)

  Estatísticas da Tabela t   
┏━━━━━━┳━━━━━━━━━┳━━━━━━━━━━┓
┃ Nome ┃ Páginas ┃   Tuplas ┃
┡━━━━━━╇━━━━━━━━━╇━━━━━━━━━━┩
│ t    │     443 │ 100000.0 │
└──────┴─────────┴──────────┘

---
## Tarefa 7
**Blocos**

Verifique quantos blocos foram efetivamente usados numa consulta

Comando:
```sql
SELECT pg_sleep(1);
\pset x on
SELECT * FROM pg_stats WHERE tablename='t';
SELECT pg_stat_reset();
\pset x off
```

- `SELECT pg_sleep(1);` faz o servidor dormir por 1 segundo; usado para sincronizar, dar tempo para o autovacuum ou apenas demonstrar espera
- `\pset x on` ativa o modo expandido do psql, exibindo cada coluna em formato vertical, facilitando a leitura quando a linha tem muitas colunas
- `SELECT * FROM pg_stats WHERE tablename='t';` mostra estatísticas coletadas pelo ANALYZE sobre colunas de tabelas, filtrando para mostrar somente informações referentes à tabela 't'
- `SELECT pg_stat_reset();` reseta estatísticas de todo o cluster, incluindo pg_stat_all_tables, desde que o usuário seja superuser. Não afeta pg_stats, pois pg_stats depende das estatísticas coletadas pelo ANALYZE
- `\pset x off` volta o psql ao modo de exibição normal (tabelado horizontal)


### O que entregar

Imprimir o resultado do comando SQL

⚠️ Os comandos foram feitos via terminal

In [35]:
!psql -d icomp -U icomp -c "SELECT pg_sleep(1);"

 pg_sleep 
----------
 
(1 row)



In [36]:
!psql -d icomp -U icomp -c "\pset x on" -c "SELECT * FROM pg_stats WHERE tablename='t';" -c "SELECT pg_stat_reset();" -c "\pset x off"

Expanded display is on.
-[ RECORD 1 ]----------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
schemaname             | public
tablename              | t
attname                | k
inherited              | f
null_frac              | 0
avg_width              | 4
n_distinct             | -1
most_common_vals       | 
most_common_freqs      | 
histogram_bounds       | {4,1037,2084,3116,4101,5003,5960,6955,7881,8806,9861,10901,11982,12907,13905,15025,15

---
## Tarefa 8
**Índice**

1. Crie um índice para o atributo 'v' e realize consultas e criação de índice

    a. Qual o tempo gasto para realizar uma consulta para um valor (lendo a tabela 100.000 tuplas)? 
    
    b. Qual o tempo gasto para recriar um índice para o atributo 'v'?

2. Remova a tabela 't' e crie novamente com 1.000.000 de tuplas

    a. Qual o tempo gasto para realizar uma consulta para um valor específico? 
    
    b. Qual o tempo gasto para recriar um índice para o atributo 'v'?

### O que entregar

**Relatório com o resultado das perguntas**

Antes de tudo, criar uma função para medir o tempo gasto em consultas

In [71]:
import time

def time_query(query, params=None) -> tuple:
    start = time.time()
    cur.execute(query, params)
    result = cur.fetchall()
    end = time.time()
    
    total_time = end - start
    return total_time, result

In [72]:
conn.rollback()
cur.execute("DROP INDEX IF EXISTS idx_t_v;")
conn.commit()

#### 1. Crie um índice para o atributo 'v' e realize consultas e criação de índice


##### **A)** Qual o tempo gasto para realizar uma consulta para um valor (lendo a tabela 100.000 tuplas)?

Sem índice

In [73]:
query = "SELECT * FROM t WHERE v = %s;"
time_no_index, rows = time_query(query, (5,))

console.print("[#A8EFFF]Tempo sem índice:[/] "
              f"[white]{time_no_index:.6f} segundos[/white]")
console.print("[#A8EFFF]Linhas retornadas:[/] "
              f"[white]{len(rows)}[/white]")

Tempo sem índice: 0.083069 segundos

Linhas retornadas: 90989

In [74]:
# Criar índice
cur.execute("CREATE INDEX IF NOT EXISTS idx_t_v ON t(v);")
conn.commit()

Com índice

In [75]:
time_with_index, rows = time_query(query, (5,))

console.print("[#A8EFFF]Tempo com índice:[/] "
              f"[white]{time_with_index:.6f} segundos[/white]")
console.print("[#A8EFFF]Linhas retornadas:[/] "
              f"[white]{len(rows)}[/white]")

Tempo com índice: 0.042238 segundos

Linhas retornadas: 90989

##### **B)** Qual o tempo gasto para recriar um índice para o atributo 'v'?

In [77]:
conn.rollback()

# Remover índice
t0 = time.perf_counter()
cur.execute("DROP INDEX IF EXISTS idx_t_v;")
conn.commit()
t1 = time.perf_counter()
drop_time = t1 - t0

# Criar novamente
t0 = time.perf_counter()
cur.execute("CREATE INDEX idx_t_v ON t(v);")
conn.commit()
t1 = time.perf_counter()
recreate_time = t1 - t0

console.print("[#A8EFFF]Tempo para DROP INDEX:[/] "
              f"[white]{drop_time:.6f} segundos[/white]")
console.print("[#A8EFFF]Tempo para recriar índice:[/] "
              f"[white]{recreate_time:.6f} segundos[/white]")

Tempo para DROP INDEX: 0.019430 segundos

Tempo para recriar índice: 0.414815 segundos

#### 2. Remova a tabela 't' e crie novamente com 1.000.000 de tuplas

In [80]:
cur.execute("""
DROP TABLE IF EXISTS t;
CREATE TABLE t (
    k SERIAL PRIMARY KEY,
    v INTEGER
);
""")

cur.execute("""
INSERT INTO t(v)
SELECT trunc(random() * 11)
FROM generate_series(1, 1000000);
""")

In [81]:
conn.rollback()
cur.execute("DROP INDEX IF EXISTS idx_t_v;")
conn.commit()

##### **A)** Qual o tempo gasto para realizar uma consulta para um valor específico?

Sem Índice

In [82]:
query = "SELECT * FROM t WHERE v = %s;"
time_no_index_1M, rows = time_query(query, (5,))

console.print("[#A8EFFF]Tempo sem índice (1.000.000 tuplas):[/] "
              f"[white]{time_no_index_1M:.6f} segundos[/white]")
console.print("[#A8EFFF]Linhas retornadas:[/] "
              f"[white]{len(rows)}[/white]")


Tempo sem índice (1.000.000 tuplas): 0.100445 segundos

Linhas retornadas: 90989

In [83]:
# Criar índice
cur.execute("CREATE INDEX IF NOT EXISTS idx_t_v ON t(v);")
conn.commit()

Com Índice

In [84]:
time_with_index_1M, rows = time_query(query, (5,))

console.print("[#A8EFFF]Tempo com índice (1.000.000 tuplas):[/] "
              f"[white]{time_with_index_1M:.6f} segundos[/white]")
console.print("[#A8EFFF]Linhas retornadas:[/] "
              f"[white]{len(rows)}[/white]")

Tempo com índice (1.000.000 tuplas): 0.068411 segundos

Linhas retornadas: 90989

##### **B)** Qual o tempo gasto para recriar um índice para o atributo 'v'?

In [85]:
# Remover índice
t0 = time.perf_counter()
cur.execute("DROP INDEX IF EXISTS idx_t_v;")
conn.commit()
t1 = time.perf_counter()
drop_time = t1 - t0

# Criar novamente
t0 = time.perf_counter()
cur.execute("CREATE INDEX idx_t_v ON t(v);")
conn.commit()
t1 = time.perf_counter()
recreate_time = t1 - t0

console.print("[#A8EFFF]Tempo para DROP INDEX (1mi tuplas):[/] "
              f"[white]{drop_time:.6f} segundos[/white]")
console.print("[#A8EFFF]Tempo para recriar índice (1mi tuplas):[/] "
              f"[white]{recreate_time:.6f} segundos[/white]")

Tempo para DROP INDEX (1mi tuplas): 0.082775 segundos

Tempo para recriar índice (1mi tuplas): 0.712117 segundos